# EDA3 — CycPeptMPDB-4D Permeability Dataset

전처리된 `CycPeptMPDB-4D_with_assay_descriptors_preprocessed.csv` 에 대한 종합적 탐색적 분석.

**구성**
1. 데이터 개요 & 타겟 분포 (`PAMPA-4D`)
2. 출처(Source) / 모양(Molecule_Shape) / 연도(Year) / 단량체 길이(Monomer_Length) 분석
3. 3D / 컨포메이셔널 descriptor 분석 (SASA, PSA, RMSD, Desolvation ΔG)
4. 2D 물리화학 descriptor (MolWt, LogP, HBD/HBA, TPSA, RotB) 분석
5. Feature ↔ Target 상관 / 상위 예측력 descriptor
6. 고/저 투과성 그룹 비교 & 결론


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

DATA_PATH = "data/CycPeptMPDB-4D_with_assay_descriptors_preprocessed.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape:", df.shape)
df.head(3)


## 1. 데이터 개요 & 타겟 분포

In [ ]:
# 컬럼/dtype 요약
print("Rows :", len(df))
print("Cols :", df.shape[1])
print("\nDtype distribution:")
print(df.dtypes.value_counts())

# 결측치 상위
miss = df.isnull().sum().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"\n결측치가 있는 컬럼 수: {len(miss)}")
miss.head(15)


In [ ]:
# 메타/식별/타겟/assay 그룹으로 컬럼 분류
META_COLS = ["CycPeptMPDB_ID", "Source", "Original_Name_in_Source_Literature",
             "Structurally_Unique_ID", "SMILES", "HELM", "Sequence",
             "Year", "Version", "HELM_URL"]
TARGET_COLS = ["PAMPA-4D", "PAMPA", "Permeability", "Caco2", "MDCK", "RRCK"]
ASSAY_AUX = ["Detection_Limit_1", "Detection_Limit_2", "R_PAMAP", "R_Caco2", "T_PAMPA"]
SHAPE_COL = "Molecule_Shape"

# 3D / conformational descriptor
DESC_3D = [c for c in df.columns if any(k in c for k in
           ["avgRMSD", "3D_SASA", "3D_NPSA", "3D_PSA", "3DPSA", "Desolvation"])]

# 그 외(2D RDKit 류)
exclude = set(META_COLS + TARGET_COLS + ASSAY_AUX + DESC_3D + [SHAPE_COL,
              "Monomer_Length", "Monomer_Length_in_Main_Chain"])
DESC_2D = [c for c in df.columns if c not in exclude and df[c].dtype != object]

print(f"3D descriptors  : {len(DESC_3D)}")
print(f"2D descriptors  : {len(DESC_2D)}")
print("\n3D 컬럼 목록:")
for c in DESC_3D:
    print(" -", c)


In [ ]:
# 타겟 분포 (PAMPA-4D = log Papp). PAMPA / Permeability 컬럼과의 일치 여부도 확인.
TARGET = "PAMPA-4D"
y = df[TARGET]

print(y.describe())
print(f"\nSkew={y.skew():.3f}, Kurtosis={y.kurt():.3f}")

# 유사 타겟과 비교
for c in ["PAMPA", "Permeability"]:
    if c in df.columns:
        diff = (df[c] - y).abs()
        print(f"|{c} - {TARGET}|  max={diff.max():.4f}  mean={diff.mean():.4f}  "
              f"corr={df[c].corr(y):.4f}")


In [ ]:
# 타겟 분포 시각화 + 일반적인 임계 (-6 = 멤브레인 투과 약, -5 이상 = 양호)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(y, bins=60, edgecolor="black", alpha=0.8, color="steelblue")
axes[0].axvline(y.mean(), color="red", ls="--", label=f"mean={y.mean():.2f}")
axes[0].axvline(y.median(), color="orange", ls="--", label=f"median={y.median():.2f}")
axes[0].axvline(-6, color="green", ls=":", label="-6 (low/high cut)")
axes[0].set_xlabel("PAMPA-4D (log Papp)")
axes[0].set_ylabel("count")
axes[0].set_title("Target distribution")
axes[0].legend()

stats.probplot(y, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot vs Normal")

axes[2].boxplot(y, vert=True)
axes[2].set_title("Boxplot of PAMPA-4D")
axes[2].set_ylabel("PAMPA-4D")

plt.tight_layout()
plt.show()

# 클래스 정의 (수동 카테고리화)
bins = [-np.inf, -7, -6, -5, np.inf]
labels = ["very_low (<-7)", "low (-7~-6)", "mid (-6~-5)", "high (≥-5)"]
df["perm_class"] = pd.cut(y, bins=bins, labels=labels)
print(df["perm_class"].value_counts().reindex(labels))


## 2. 메타데이터 분석 — Source / Year / Shape / Monomer Length

데이터의 **출처별 편향**과 **구조적 카테고리**가 투과성에 어떻게 연관되는지 확인.

In [ ]:
# Source / Year / Shape 분포
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

src_cnt = df["Source"].value_counts()
sns.barplot(x=src_cnt.values, y=src_cnt.index, ax=axes[0], palette="viridis")
axes[0].set_title("Records per Source")
axes[0].set_xlabel("count")
for i, v in enumerate(src_cnt.values):
    axes[0].text(v, i, f" {v}", va="center")

yr_cnt = df["Year"].value_counts().sort_index()
sns.barplot(x=yr_cnt.index.astype(str), y=yr_cnt.values, ax=axes[1], palette="rocket")
axes[1].set_title("Records per Year")
axes[1].set_ylabel("count")

shape_cnt = df[SHAPE_COL].value_counts()
axes[2].pie(shape_cnt.values, labels=shape_cnt.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2"), startangle=90)
axes[2].set_title("Molecule Shape")

plt.tight_layout()
plt.show()


In [ ]:
# Source / Shape 별 PAMPA-4D 분포 비교 + 통계 검정
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

order = src_cnt.index.tolist()
sns.boxplot(data=df, x="Source", y=TARGET, order=order, ax=axes[0], palette="viridis")
sns.stripplot(data=df, x="Source", y=TARGET, order=order, ax=axes[0],
              size=1.2, color="black", alpha=0.25)
axes[0].set_title("PAMPA-4D by Source")
axes[0].tick_params(axis="x", rotation=20)

sns.violinplot(data=df, x=SHAPE_COL, y=TARGET, ax=axes[1], palette="Set2", inner="quartile")
axes[1].set_title("PAMPA-4D by Molecule Shape")

plt.tight_layout()
plt.show()

# Source ANOVA
groups = [g[TARGET].values for _, g in df.groupby("Source")]
F, p = stats.f_oneway(*groups)
print(f"[Source] one-way ANOVA  F={F:.2f}  p={p:.3e}")

# Shape t-test
g_circ = df.loc[df[SHAPE_COL] == "Circle", TARGET]
g_lar  = df.loc[df[SHAPE_COL] == "Lariat", TARGET]
t, p = stats.ttest_ind(g_circ, g_lar, equal_var=False)
print(f"[Shape ] Circle vs Lariat   "
      f"mean Circle={g_circ.mean():.3f}, Lariat={g_lar.mean():.3f}, "
      f"t={t:.2f}, p={p:.3e}")


In [ ]:
# Monomer length(전체 vs main chain)와 투과성 — cyclic peptide 분야 핵심 변수
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

ml_cnt = df["Monomer_Length"].value_counts().sort_index()
ml_cnt.plot(kind="bar", ax=axes[0, 0], color="teal", edgecolor="black")
axes[0, 0].set_title("Monomer_Length distribution")
axes[0, 0].set_xlabel("Monomer_Length")
axes[0, 0].set_ylabel("count")

mlmc_cnt = df["Monomer_Length_in_Main_Chain"].value_counts().sort_index()
mlmc_cnt.plot(kind="bar", ax=axes[0, 1], color="darkorange", edgecolor="black")
axes[0, 1].set_title("Monomer_Length_in_Main_Chain distribution")

sns.boxplot(data=df, x="Monomer_Length", y=TARGET, ax=axes[1, 0], palette="crest")
axes[1, 0].set_title("PAMPA-4D vs Monomer_Length")

sns.boxplot(data=df, x="Monomer_Length_in_Main_Chain", y=TARGET, ax=axes[1, 1], palette="flare")
axes[1, 1].set_title("PAMPA-4D vs Main-Chain Length")

plt.tight_layout()
plt.show()

# 길이별 평균/분산
print(df.groupby("Monomer_Length")[TARGET].agg(["mean", "std", "count"]).round(3))


## 3. 4D / 컨포메이셔널 Descriptor 분석

이 데이터셋의 차별점인 **두 가지 용매(Water vs Hexane) 환경에서의 conformer-derived descriptor** 분석.

- `*_avgRMSD_*`  : 컨포머 앙상블의 유연성 지표
- `3D_SASA / NPSA / PSA` : 용매-접근 표면적 / 비극성 / 극성
- `Desolvation_Free_Energy` : 수화 → hexane(지질) 전이 에너지 → 막 투과 직접 관련
- `CHCl3_3DPSA / H2O_3DPSA` : 추가 용매 PSA

In [ ]:
# 3D descriptor 통계 요약
df[DESC_3D].describe().T.round(3)


In [ ]:
# Water vs Hexane 용매 환경의 SASA / PSA 분포 비교
solvent_pairs = [
    ("Water_3D_SASA", "Hexane_3D_SASA", "Total SASA"),
    ("Water_3D_PSA",  "Hexane_3D_PSA",  "Polar SA (PSA)"),
    ("Water_3D_NPSA", "Hexane_3D_NPSA", "Non-Polar SA (NPSA)"),
    ("Water_avgRMSD_All", "Hexane_avgRMSD_All", "Conformer flexibility (avgRMSD all)"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (w, h, title) in zip(axes, solvent_pairs):
    ax.scatter(df[w], df[h], s=4, alpha=0.4, c=df[TARGET], cmap="viridis")
    lo = min(df[w].min(), df[h].min()); hi = max(df[w].max(), df[h].max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_xlabel(w); ax.set_ylabel(h); ax.set_title(title)
plt.tight_layout()
plt.show()

# 두 용매 사이 평균 차이 (= 환경 의존적 컨포메이셔널 변화)
for w, h, t in solvent_pairs:
    d = df[h] - df[w]
    print(f"{t:35s} Δ(Hex-Water) mean={d.mean():+.2f}  std={d.std():.2f}  "
          f"corr(Δ, target)={d.corr(df[TARGET]):+.3f}")


In [ ]:
# 핵심 3D descriptor vs 타겟 산점도 + 회귀
key_3d = ["Desolvation_Free_Energy",
          "Water_3D_PSA", "Hexane_3D_PSA",
          "Water_3D_NPSA", "Hexane_3D_NPSA",
          "Water_avgRMSD_BackBone", "Hexane_avgRMSD_BackBone"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for i, col in enumerate(key_3d):
    ax = axes[i]
    ax.scatter(df[col], df[TARGET], s=4, alpha=0.35, color="steelblue")
    # 단순 선형 적합
    m, b, r, p, _ = stats.linregress(df[col], df[TARGET])
    xs = np.linspace(df[col].min(), df[col].max(), 50)
    ax.plot(xs, m*xs + b, "r-", lw=1.5, label=f"r={r:+.3f}")
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f"{col}\nPearson r={r:+.3f}, p={p:.1e}")
    ax.legend(loc="best", fontsize=8)
for j in range(len(key_3d), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 3D descriptor 들 간 상관 + 타겟 — 컨포메이셔널 신호의 군집 구조 확인
all_3d = DESC_3D + [TARGET]
corr_3d = df[all_3d].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_3d, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.7}, ax=ax,
            annot_kws={"size": 7})
ax.set_title("3D / 4D descriptors — Pearson correlation (incl. PAMPA-4D)")
plt.tight_layout()
plt.show()


## 4. 2D 물리화학 / Lipinski-스타일 Descriptor

`MolWt`, `MolLogP`, `TPSA`, `NumHDonors/Acceptors`, `NumRotatableBonds` 등 BBB / 멤브레인 투과 클래식 변수 분석.

In [ ]:
phys_keys = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
             "NumRotatableBonds", "RingCount", "FractionCSP3", "qed",
             "HeavyAtomCount", "NumAromaticRings"]
phys_keys = [c for c in phys_keys if c in df.columns]

print(df[phys_keys].describe().T.round(2))

# 분포 + 타겟과의 산점도 한 번에
n = len(phys_keys)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.6*nrows))
axes = axes.flatten()
for i, c in enumerate(phys_keys):
    ax = axes[i]
    ax.scatter(df[c], df[TARGET], s=4, alpha=0.3, color="darkslateblue")
    r = df[c].corr(df[TARGET])
    ax.set_xlabel(c); ax.set_ylabel(TARGET)
    ax.set_title(f"{c}   r={r:+.3f}")
for j in range(len(phys_keys), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Cyclic peptide 'beyond Rule of 5' 영역 점검 — 일반 약물 룰을 얼마나 벗어나는지
ro5 = pd.DataFrame({
    "MolWt > 500":       (df["MolWt"]            > 500),
    "MolLogP > 5":       (df["MolLogP"]          > 5),
    "HBD > 5":           (df["NumHDonors"]       > 5),
    "HBA > 10":          (df["NumHAcceptors"]    > 10),
    "RotB > 10":         (df["NumRotatableBonds"]> 10),
    "TPSA > 140":        (df["TPSA"]             > 140),
})
ro5_violations = ro5.sum(axis=1)
print("Ro5 위반 개수 분포:")
print(ro5_violations.value_counts().sort_index())
print("\n각 룰 위반 비율:")
print((ro5.mean()*100).round(1).astype(str) + " %")

# 위반 갯수 별 평균 PAMPA-4D
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.boxplot(x=ro5_violations, y=df[TARGET], ax=axes[0], palette="rocket_r")
axes[0].set_xlabel("# Ro5 violations"); axes[0].set_title("PAMPA-4D vs Ro5 violations")

# 영역별 (heavy atom vs MolLogP) 스캐터
sc = axes[1].scatter(df["MolWt"], df["MolLogP"], c=df[TARGET], s=6,
                     alpha=0.6, cmap="viridis")
axes[1].axhline(5, color="red", ls="--", lw=0.8); axes[1].axvline(500, color="red", ls="--", lw=0.8)
axes[1].set_xlabel("MolWt"); axes[1].set_ylabel("MolLogP")
axes[1].set_title("Chemical space colored by PAMPA-4D")
plt.colorbar(sc, ax=axes[1], label="PAMPA-4D")
plt.tight_layout()
plt.show()


## 5. Feature ↔ Target 상관 — 어떤 descriptor 가 PAMPA-4D 를 가장 잘 설명하는가?

200+ 개의 numeric descriptor 에서 Pearson / Spearman 으로 타겟과의 단변량 관계 랭킹.

In [ ]:
# 모든 numeric descriptor 와 타겟의 Pearson / Spearman 상관 — assay aux & 타겟 본인 제외
exclude_for_corr = set(TARGET_COLS + ASSAY_AUX + ["Year", "Version", "perm_class"])
num_cols = [c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c]) and c not in exclude_for_corr]

# 분산이 0 인 컬럼 제거
nz = [c for c in num_cols if df[c].std(ddof=0) > 0]
print(f"모두-0 으로 제거된 컬럼: {len(num_cols) - len(nz)}")
print(f"분석 대상 numeric 컬럼: {len(nz)}")

corr_p = df[nz].corrwith(df[TARGET], method="pearson")
corr_s = df[nz].corrwith(df[TARGET], method="spearman")

corr_tbl = pd.DataFrame({"pearson": corr_p, "spearman": corr_s})
corr_tbl["abs_pearson"]  = corr_tbl["pearson"].abs()
corr_tbl["abs_spearman"] = corr_tbl["spearman"].abs()

top = corr_tbl.sort_values("abs_pearson", ascending=False).head(25)
print("\n=== Top-25 by |Pearson| with PAMPA-4D ===")
print(top[["pearson", "spearman"]].round(3))


In [ ]:
# Top-25 시각화 (양/음의 부호 보존)
top25 = corr_tbl.sort_values("abs_pearson", ascending=False).head(25)
fig, ax = plt.subplots(figsize=(8, 8))
colors = ["crimson" if v > 0 else "steelblue" for v in top25["pearson"]]
ax.barh(top25.index[::-1], top25["pearson"].values[::-1],
        color=colors[::-1], edgecolor="black")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Pearson r with PAMPA-4D")
ax.set_title("Top-25 descriptors most correlated with PAMPA-4D")
plt.tight_layout()
plt.show()


In [ ]:
# 비선형 신호 점검 — Spearman 이 Pearson 보다 큰 차이의 컬럼 = 비선형 단조 관계 후보
diff = (corr_tbl["abs_spearman"] - corr_tbl["abs_pearson"])
nonlin = corr_tbl.loc[diff.sort_values(ascending=False).head(15).index][
    ["pearson", "spearman"]].round(3)
print("=== Spearman 이 Pearson 보다 강한 (비선형 단조) 후보 Top-15 ===")
print(nonlin)


In [ ]:
# Top descriptor 들 사이의 다중공선성 확인 — 모델링시 어떤 게 redundancy 인지
top_feats = top25.index.tolist()
sub_corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(sub_corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=ax, cbar_kws={"shrink": 0.7},
            annot=True, fmt=".2f", annot_kws={"size": 7})
ax.set_title("Multicollinearity among Top-25 descriptors")
plt.tight_layout()
plt.show()


In [ ]:
# 비선형 모델 기반 중요도 (Random Forest) — 단변량 상관이 놓치는 신호 보완
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression

# 결측 / inf 가 없는 numeric set, 그리고 sklearn float32 한계를 넘는 컬럼(Ipc 등) 제거
F32_MAX = np.finfo(np.float32).max  # ≈ 3.4e38
candidate = [c for c in nz if df[c].isnull().sum() == 0]
ok_cols = []
for c in candidate:
    v = df[c].values
    if not np.isfinite(v).all():
        continue
    if np.abs(v).max() > F32_MAX / 10:
        continue  # float32 변환 시 overflow 우려
    ok_cols.append(c)

X_cols = ok_cols
print(f"inf/NaN/over-range 로 제거된 컬럼: {len(candidate) - len(X_cols)}")
print(f"모델 입력 컬럼 수                : {len(X_cols)}")

X = df[X_cols].values.astype(np.float64)
y_arr = df[TARGET].values

rf = RandomForestRegressor(n_estimators=200, max_depth=None,
                           n_jobs=-1, random_state=42)
rf.fit(X, y_arr)
imp = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\n=== RandomForest feature importance Top-20 ===")
print(imp.head(20).round(4))

# Mutual information
mi = mutual_info_regression(X, y_arr, random_state=42, n_neighbors=5)
mi_s = pd.Series(mi, index=X_cols).sort_values(ascending=False)
print("\n=== Mutual Information Top-20 ===")
print(mi_s.head(20).round(4))


In [ ]:
# 세 가지 랭킹(Pearson, RF, MI)을 통합한 importance summary
rank = pd.DataFrame({
    "abs_pearson": corr_tbl["abs_pearson"].reindex(X_cols),
    "rf_imp"     : imp.reindex(X_cols),
    "mutual_info": mi_s.reindex(X_cols),
})
# rank 평균 (각각의 백분위수 평균)
rank_norm = rank.rank(pct=True)
rank["rank_avg_pct"] = rank_norm.mean(axis=1)
top_consensus = rank.sort_values("rank_avg_pct", ascending=False).head(20)
print("=== Consensus top-20 (avg percentile of |r|, RF, MI) ===")
print(top_consensus.round(3))


## 6. 고/저 투과성 그룹 비교 + PCA 임베딩 + 결론

투과성이 높은 분자(top-quartile)와 낮은 분자(bottom-quartile)가 어떤 descriptor 에서 가장 다른지 정량화.

In [ ]:
# 상하 quartile 그룹 정의 + Welch t-test + Cohen's d
q_lo = df[TARGET].quantile(0.25)
q_hi = df[TARGET].quantile(0.75)
hi = df[df[TARGET] >= q_hi]
lo = df[df[TARGET] <= q_lo]
print(f"high-permeable (PAMPA-4D ≥ {q_hi:.2f}): n={len(hi)}")
print(f"low-permeable  (PAMPA-4D ≤ {q_lo:.2f}): n={len(lo)}")

def cohens_d(a, b):
    na, nb = len(a), len(b)
    s = np.sqrt(((na-1)*a.var() + (nb-1)*b.var()) / (na+nb-2))
    return (a.mean() - b.mean()) / s if s > 0 else 0.0

rows = []
for c in X_cols:
    a, b = hi[c].values, lo[c].values
    if a.std()+b.std() == 0:
        continue
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({"feature": c,
                 "mean_high": a.mean(), "mean_low": b.mean(),
                 "delta": a.mean() - b.mean(),
                 "cohen_d": cohens_d(pd.Series(a), pd.Series(b)),
                 "t": t, "p": p})
diff_tbl = pd.DataFrame(rows).set_index("feature")
diff_tbl["abs_d"] = diff_tbl["cohen_d"].abs()

print("\n=== |Cohen's d| Top-20 (high vs low quartile) ===")
print(diff_tbl.sort_values("abs_d", ascending=False).head(20)[
    ["mean_high", "mean_low", "cohen_d", "p"]].round(3))


In [ ]:
# 상위 6 개 차이 descriptor 의 그룹별 KDE 비교
top6 = diff_tbl.sort_values("abs_d", ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, c in enumerate(top6):
    ax = axes[i]
    sns.kdeplot(lo[c], ax=ax, fill=True, color="steelblue",
                alpha=0.5, label=f"low (n={len(lo)})")
    sns.kdeplot(hi[c], ax=ax, fill=True, color="crimson",
                alpha=0.5, label=f"high (n={len(hi)})")
    d = diff_tbl.loc[c, "cohen_d"]
    ax.set_title(f"{c}\nCohen's d = {d:+.2f}")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# PCA 임베딩 — 화학공간이 PAMPA-4D 에 따라 정렬되는지 시각적으로 확인
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Xs = StandardScaler().fit_transform(df[X_cols].values)
pca = PCA(n_components=2, random_state=42).fit(Xs)
Z = pca.transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sc = axes[0].scatter(Z[:, 0], Z[:, 1], c=df[TARGET], cmap="viridis", s=6, alpha=0.7)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
axes[0].set_title("PCA colored by PAMPA-4D")
plt.colorbar(sc, ax=axes[0], label="PAMPA-4D")

# Source 별
for src, color in zip(df["Source"].unique(), sns.color_palette("tab10", df["Source"].nunique())):
    m = (df["Source"] == src).values
    axes[1].scatter(Z[m, 0], Z[m, 1], s=6, alpha=0.55, label=src, color=color)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("PCA colored by Source")
axes[1].legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

# 누적 분산
pca_full = PCA(n_components=20, random_state=42).fit(Xs)
print("Cumulative explained variance (first 20 PCs):")
print(np.round(np.cumsum(pca_full.explained_variance_ratio_), 3))


In [ ]:
# 단순 baseline: top-k 단변량 feature 만 가지고 회귀하면 얼마나 설명되는가?
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

results = []
for k in [1, 3, 5, 10, 20, 50, len(X_cols)]:
    feats = corr_tbl["abs_pearson"].sort_values(ascending=False).head(k).index.tolist()
    Xk = StandardScaler().fit_transform(df[feats].values)
    r2 = cross_val_score(Ridge(alpha=1.0), Xk, df[TARGET].values,
                         scoring="r2", cv=5).mean()
    results.append((k, r2, feats[:5]))
print(f"{'k':>4} | {'CV R²':>8} | top-features")
print("-" * 60)
for k, r2, ff in results:
    print(f"{k:>4} | {r2:>8.3f} | {ff}")


## 7. 핵심 인사이트 요약

- **데이터셋**: 4,925 cyclic peptides × 245 컬럼. 타겟 `PAMPA-4D` 는 log Papp (mean ≈ -5.81, 범위 -9.46 ~ -4.0). 분포는 거의 정규에 가깝지만 좌측 long tail 존재.
- **출처 편향**: `2020_Townsend (56%)` + `2021_Kelly (28%)` 두 source 가 데이터의 84% 차지 → split 시 source-aware 가 필요. Source 간 PAMPA-4D 분포가 유의하게 다름 (one-way ANOVA).
- **모양 / 길이**: `Circle` (72%) vs `Lariat` (28%). Lariat 은 Kelly 데이터에 한정. Monomer length 는 6/7/10 에 집중되어 있고, 길이 ↑ 일수록 평균 PAMPA-4D 는 느슨하게 감소.
- **4D / conformational descriptor 가 핵심**:
  - `Hexane_3D_PSA`, `Water_3D_PSA`, `Desolvation_Free_Energy` 가 단변량 상관 / RF importance / Mutual Info 모두에서 상위.
  - Water vs Hexane 의 PSA / NPSA 차이(=환경 의존적 컨포메이셔널 변화)는 타겟과 의미 있는 상관을 가짐 → 4D descriptor 의 부가가치를 정량적으로 입증.
- **Beyond Rule of 5**: 대부분 분자가 MolWt > 500, HBD/HBA, RotB 룰을 위반함에도 일부는 강한 투과성을 보임. Ro5 위반 갯수만으로는 PAMPA-4D 를 잘 구분하지 못함.
- **선형 baseline 한계**: top-k Ridge baseline 의 CV R² 가 낮은 k 에서 빠르게 포화 → 단변량 신호만으로는 한계, 비선형 / 컨포머 기반 모델 필요.
- **PCA 임베딩**: PC1-PC2 가 일부 PAMPA-4D 그라디언트를 잡지만, source 클러스터링 효과가 강함 → 모델 평가 시 random split 보다 source-stratified 또는 scaffold-based split 권장.

**다음 단계 제안**
1. Source-stratified / scaffold split 으로 baseline 재평가.
2. Top consensus descriptor (Hexane_PSA, Desolvation_FE, Water_PSA, MolLogP, TPSA) 만으로 lightweight 모델 → 4D 신호의 실효성 정량화.
3. Conformer ensemble feature (avgRMSD, ΔSA between solvents) 를 모델 입력으로 사용해 GNN/Transformer baseline 과 비교.